In [2]:
# Install Gradio
!pip install gradio

import re
import gradio as gr

# -----------------------------
# 1. SECURITY VALIDATION
# -----------------------------

DANGEROUS_KEYWORDS = [
    "drop", "delete", "alter", "truncate",
    "insert", "update", "exec", "execute"
]

DANGEROUS_PATTERNS = [
    ";", "--", "/*", "*/", "xp_", "sp_"
]

def is_safe(user_input):
    lower_input = user_input.lower()

    for word in DANGEROUS_KEYWORDS:
        if word in lower_input:
            return False

    for pattern in DANGEROUS_PATTERNS:
        if pattern in lower_input:
            return False

    return True


# -----------------------------
# 2. TABLE DETECTION
# -----------------------------

def get_table_name(text):
    text = text.lower()

    if any(word in text for word in ["user", "users", "customer", "account"]):
        return "users"

    if any(word in text for word in ["order", "orders", "purchase"]):
        return "orders"

    if any(word in text for word in ["product", "products", "item"]):
        return "products"

    if any(word in text for word in ["transaction", "transactions", "payment"]):
        return "transactions"

    return None


# -----------------------------
# 3. CONDITION EXTRACTION
# -----------------------------

def extract_conditions(text, table):
    text = text.lower()
    conditions = []

    amount_match = re.search(r"greater than (\d+)", text)
    if amount_match:
        amount = amount_match.group(1)
        conditions.append(f"amount > {amount}")

    if "last week" in text:
        if table == "transactions":
            conditions.append("DATE(date) >= DATE('now', '-7 days')")
        elif table == "users":
            conditions.append("DATE(signup_date) >= DATE('now', '-7 days')")

    if "last month" in text:
        if table == "users":
            conditions.append("DATE(signup_date) >= DATE('now', '-1 month')")

    return conditions


# -----------------------------
# 4. SQL GENERATOR
# -----------------------------

def generate_sql(user_input):

    if not is_safe(user_input):
        return "Unsafe query detected."

    table = get_table_name(user_input)

    if not table:
        return "Could not detect table name."

    conditions = extract_conditions(user_input, table)

    sql = f"SELECT * FROM {table}"

    if conditions:
        sql += " WHERE " + " AND ".join(conditions)

    sql += " LIMIT 10"

    return sql


# -----------------------------
# 5. WEBSITE UI
# -----------------------------

interface = gr.Interface(
    fn=generate_sql,
    inputs=gr.Textbox(label="Enter Natural Language Query"),
    outputs=gr.Textbox(label="Generated SQL Query"),
    title="Natural Language to SQL Generator",
    description="Type a query like: Find transactions greater than 500 from last week"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bc0327b64a72d9565b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
